# 01-02 - Stanford Online Products - Dataset Preparation & Stratified Sampling
___

Mục tiêu:

1. Clone repository chứa code/project.
2. Tải Stanford Online Products từ Google Drive.
3. Kiểm tra cấu trúc dataset.
4. Parse `Ebay_train.txt` và `Ebay_test.txt`.
5. Kiểm tra train/test class disjointness.
6. Stratified sampling:
   - Train: 10,000 images
   - Test: 10,000 images
7. Giữ nguyên tỷ lệ phân bố class gần với dataset gốc.
8. Kiểm tra class coverage, image coverage và zero-shot constraint.
9. Xuất dataset metadata để sử dụng cho preprocessing và DINOv3.

Lưu ý:
- Train/test của SOP có disjoint class_id.
- Không được gộp train/test trước khi sampling.
- Mỗi class có thể có số lượng ảnh khác nhau.

## 1. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
from pathlib import Path

# Project
PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data"
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
SAMPLE_DIR = DATA_ROOT / "sampled"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# Git repository
GIT_REPO_URL = "https://github.com/HCMUS-VIR-Nhom8/DINOv3-FAISS-HNSW-SOP-VisualProductSearch.git"
GIT_BRANCH = "sop-pre-processing"

CLONE_REPO = True

# Google Drive
GOOGLE_DRIVE_URL = "https://drive.google.com/drive/folders/1uGmuwlITW5gffgUQxoyXAQtIa2y_LuYM?usp=sharing"
GOOGLE_DRIVE_FILE_ID = "1y2FefjA1IGXrop8aJqcNMjFVIPR4o8dm" # Nếu sử dụng gdown
DOWNLOAD_DATASET = True

# Dataset
DATASET_NAME = "Stanford_Online_Products"
DATASET_DIR = RAW_DIR / DATASET_NAME
TRAIN_META = DATASET_DIR / "Ebay_train.txt"
TEST_META = DATASET_DIR / "Ebay_test.txt"

# Sampling
TOTAL_SAMPLE_SIZE = 20_000
TRAIN_SAMPLE_SIZE = 10_000
TEST_SAMPLE_SIZE = 10_000

RANDOM_SEED = 42

# Expected statistics
EXPECTED_TRAIN_IMAGES = 59_551
EXPECTED_TEST_IMAGES = 60_502
EXPECTED_TRAIN_CLASSES = 11_318
EXPECTED_TEST_CLASSES = 11_316

## Install dependencies

In [ ]:
%pip install -q gdown pandas numpy tqdm scikit-learn

Nếu chạy trên colab, có thể thêm:

In [ ]:
!apt-get -qq update
!apt-get -qq install git

## Clone repo

In [ ]:
import subprocess
from pathlib import Path

def run_command(command):
    print("Running:")
    print(" ".join(command))
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")

REPO_NAME = GIT_REPO_URL.rstrip("/").split("/")[-1]

if REPO_NAME.endswith(".git"):
    REPO_NAME = REPO_NAME[:-4]

REPO_DIR = PROJECT_ROOT / REPO_NAME

if CLONE_REPO:
    if REPO_DIR.exists():
        print(f"Repository already exists: {REPO_DIR}")
    else:
        run_command([
            "git",
            "clone",
            "--branch",
            GIT_BRANCH,
            GIT_REPO_URL,
            str(REPO_DIR)
        ])

## Tải dataset từ GGDrive

Trong drive, dataset đc nén thành `Stanford_Online_Products.zip`

In [ ]:
import gdown

DATASET_ZIP = RAW_DIR / "Stanford_Online_Products.zip"
if DOWNLOAD_DATASET:
    if not DATASET_ZIP.exists():
        gdown.download(
            id=GOOGLE_DRIVE_FILE_ID,
            output=str(DATASET_ZIP),
            quiet=False
        )
    else:
        print("Dataset archive already exists.")

In [ ]:
import zipfile
if DATASET_ZIP.exists():
    print("Extracting dataset...")
    with zipfile.ZipFile(DATASET_ZIP, "r") as zip_ref:
        zip_ref.extractall(RAW_DIR)
    print("Extraction completed.")
else:
    print("Dataset ZIP not found.")

In [ ]:
print("Dataset directory:")
print(DATASET_DIR)
print("\nFiles:")
for path in DATASET_DIR.iterdir():
    print(path)

assert TRAIN_META.exists(), (
    f"Missing: {TRAIN_META}"
)
assert TEST_META.exists(), (
    f"Missing: {TEST_META}"
)

## 2. Parse metadata (nếu cần)

Metadat có 4 thành phần:
`image_id class_id super_class_id path`

VD: 
`1 2 1 bicycle_final/111085122871_0.JPG`

In [ ]:
import pandas as pd

COLUMNS = [
    "image_id",
    "class_id",
    "super_class_id",
    "path"
]

def load_sop_metadata(meta_file):
    df = pd.read_csv(
        meta_file,
        sep=r"\s+",
        header=None,
        names=COLUMNS
    )
    return df

train_df = load_sop_metadata(TRAIN_META)
test_df = load_sop_metadata(TEST_META)

print("Train:")
print(train_df.head())
print("\nTest:")
print(test_df.head())

## Kiểm tra các statistics của dataset

In [ ]:
def dataset_statistics(df, name):
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(f"Images      : {len(df):,}")
    print(f"Classes     : {df.class_id.nunique():,}")
    print(f"Superclasses: {df.super_class_id.nunique():,}")
    
    images_per_class = df.groupby("class_id").size()

    print(
        f"Images/class:"
        f" min={images_per_class.min()},"
        f" max={images_per_class.max()},"
        f" mean={images_per_class.mean():.2f},"
        f" median={images_per_class.median():.2f}"
    )

dataset_statistics(train_df, "TRAIN")
dataset_statistics(test_df, "TEST")

## Check zero-shot class disjointness
 
Bài toán mà chúng ta làm phải là Zero-Shot Learning, tức là phải có đặc điểm: tập các class dùng để train và tập các class dùng để test phải hoàn toàn tách biệt (disjoint) ($C_{train} \cap C_{test}=\emptyset$).

Tại sao cần disjoint?  

Nếu tập test trùng lớp với tập train, mô hình chỉ cần "học vẹt" (memorize) hình dáng của các sản phẩm cụ thể đó. Khi tách biệt hoàn toàn (disjoint), chúng ta mới đánh giá được mô hình có khả năng tổng quát hóa (generalization) sang những sản phẩm hoàn toàn mới, chưa từng thấy bao giờ hay không.



In [ ]:
train_classes = set(train_df["class_id"].unique())
test_classes = set(test_df["class_id"].unique())
overlap = train_classes.intersection(test_classes)

print(f"Train classes: {len(train_classes):,}")
print(f"Test classes : {len(test_classes):,}")
print(f"Overlapping classes: {len(overlap):,}")

assert len(overlap) == 0, ("ERROR: Train/Test classes are not disjoint!")
print("✓ Zero-shot constraint verified.")

## Kiểm tra image ID Overlap

In [ ]:
train_images = set(train_df["image_id"])
test_images = set(test_df["image_id"])
image_overlap = train_images.intersection(test_images)

print(f"Overlapping image IDs: {len(image_overlap)}")
assert len(image_overlap) == 0

## Phân bố số ảnh/class

In [ ]:
import matplotlib.pyplot as plt

# tập train
train_counts = (
    train_df
    .groupby("class_id")
    .size()
)
plt.figure(figsize=(10, 5))
plt.hist(
    train_counts,
    bins=30
)
plt.xlabel("Number of images per class")
plt.ylabel("Number of classes")
plt.title("SOP Train: Images per Class")
plt.show()

# tập test
test_counts = (
    test_df
    .groupby("class_id")
    .size()
)
plt.figure(figsize=(10, 5))
plt.hist(
    test_counts,
    bins=30
)
plt.xlabel("Number of images per class")
plt.ylabel("Number of classes")
plt.title("SOP Test: Images per Class")
plt.show()

## 3. Hàm stratified sampling

Ở đây có một vấn đề là nếu chỉ dùng: `df.groupby("class_id").sample(...)` thì không xử lý tốt trường hợp class có số ảnh khác nhau.

Ta dùng proportional allocation: $n_c=round(N_{target}\frac{N_c}{N})$

In [ ]:
import numpy as np

def allocate_samples(class_counts,target_size,min_per_class=0):
    """
    Allocate samples to classes based on their counts.
    Args:
        class_counts (pd.Series): Number of images per class.
        target_size (int): Total number of samples to allocate.
        min_per_class (int): Minimum number of samples per class.
    Returns:
        pd.Series: Number of samples allocated to each class.
    """
    total = class_counts.sum()
    raw = (
        class_counts / total
    ) * target_size
    allocation = np.floor(raw).astype(int)
    if min_per_class > 0:
        allocation = np.maximum(
            allocation,
            min_per_class
        )
        allocation = np.minimum(
            allocation,
            class_counts
        )
    # Remaining samples
    current = allocation.sum()
    remaining = target_size - current
    if remaining > 0:
        fractional = raw - np.floor(raw)
        order = fractional.sort_values(
            ascending=False
        ).index
        for cls in order:
            if remaining == 0:
                break
            if allocation.loc[cls] < class_counts.loc[cls]:
                allocation.loc[cls] += 1
                remaining -= 1
    # Too many samples
    elif remaining < 0:
        fractional = raw - np.floor(raw)
        order = fractional.sort_values(
            ascending=True
        ).index
        for cls in order:
            if remaining == 0:
                break
            if allocation.loc[cls] > min_per_class:
                allocation.loc[cls] -= 1
                remaining += 1
    assert allocation.sum() == target_size
    return allocation

In [ ]:
def stratified_sample(df,target_size,seed=42):
    """
    Perform stratified sampling on a DataFrame based on class_id.
    Args:
        df (pd.DataFrame): Input DataFrame with a "class_id" column.
        target_size (int): Total number of samples to allocate.
        seed (int): Random seed for reproducibility.
    Returns:
        tuple: A tuple containing the sampled DataFrame and the allocation counts.
    """
    rng = np.random.RandomState(seed)

    class_counts = (
        df.groupby("class_id")
        .size()
    )

    allocation = allocate_samples(
        class_counts,
        target_size
    )

    sampled_parts = []

    for class_id, n_samples in allocation.items():
        class_df = df[
            df["class_id"] == class_id
        ]
        if n_samples == 0:
            continue
        sampled = class_df.sample(
            n=n_samples,
            random_state=seed
        )
        sampled_parts.append(sampled)
        
    sampled_df = pd.concat(
        sampled_parts,
        ignore_index=True
    )

    # Shuffle final dataset
    sampled_df = sampled_df.sample(
        frac=1.0,
        random_state=seed
    ).reset_index(drop=True)

    return sampled_df, allocation

In [ ]:
# Sampling train/test independently
sampled_train, train_allocation = (
    stratified_sample(
        train_df,
        TRAIN_SAMPLE_SIZE,
        RANDOM_SEED
    )
)

sampled_test, test_allocation = (
    stratified_sample(
        test_df,
        TEST_SAMPLE_SIZE,
        RANDOM_SEED
    )
)

In [ ]:
print(f"Sampled train: {len(sampled_train):,}")
print(f"Sampled test : {len(sampled_test):,}")
print(f"Total        : {len(sampled_train) + len(sampled_test):,}")

## Tính class coverage

Class coverage là ...


In [ ]:
def coverage_report(original,sampled,name):
    original_classes = set(
        original.class_id
    )
    sampled_classes = set(
        sampled.class_id
    )
    coverage = (
        len(sampled_classes)
        /
        len(original_classes)
    )
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(f"Original classes : {len(original_classes):,}")
    print(f"Sampled classes  : {len(sampled_classes):,}")
    print(f"Class coverage   : {coverage * 100:.2f}%")
    print(f"Classes lost     : {len(original_classes - sampled_classes):,}")

    return coverage

train_coverage = coverage_report(
    train_df,
    sampled_train,
    "TRAIN"
)

test_coverage = coverage_report(
    test_df,
    sampled_test,
    "TEST"
)

## Zero-shot check sau sampling

In [ ]:
sampled_train_classes = set(
    sampled_train.class_id
)

sampled_test_classes = set(
    sampled_test.class_id
)

sampled_overlap = (
    sampled_train_classes
    &
    sampled_test_classes
)

print("Sampled train/test class overlap:",len(sampled_overlap))
assert len(sampled_overlap) == 0
print("✓ Sampled dataset preserves, zero-shot class separation.")

## So sánh class distribution

In [ ]:
def class_distribution(df):
    counts = (
        df.groupby("class_id")
        .size()
    )
    return counts / len(df)

train_original_dist = class_distribution(train_df)
train_sampled_dist = class_distribution(sampled_train)
train_comparison = pd.DataFrame({
    "original": train_original_dist,
    "sampled": train_sampled_dist
}).fillna(0)
train_comparison.head()

test_original_dist = class_distribution(test_df)
test_sampled_dist = class_distribution(sampled_test)
test_comparison = pd.DataFrame({
    "original": test_original_dist,
    "sampled": test_sampled_dist
}).fillna(0)   
test_comparison.head()

## KL divergence

In [ ]:
from scipy.stats import entropy

def kl_divergence(original_dist,sampled_dist):
    all_classes = (
        set(original_dist.index)
        |
        set(sampled_dist.index)
    )

    p = original_dist.reindex(
        all_classes,
        fill_value=0
    ).values

    q = sampled_dist.reindex(
        all_classes,
        fill_value=0
    ).values

    # smoothing
    eps = 1e-12

    p = p + eps
    q = q + eps

    p = p / p.sum()
    q = q / q.sum()

    return entropy(p, q)


train_kl = kl_divergence(train_original_dist,train_sampled_dist)
print(f"Train KL divergence: {train_kl:.6f}")

test_kl = kl_divergence(test_original_dist,test_sampled_dist)
print(f"Test KL divergence: {test_kl:.6f}")

## Kiểm tra image path

In [ ]:
def check_image_paths(df,dataset_root,max_missing_report=20):
    missing = []
    for path in df["path"]:
        full_path = (dataset_root / path)
        if not full_path.exists():
            missing.append(str(full_path))
    print(f"Missing images: {len(missing):,}")
    for path in missing[:max_missing_report]:
        print(path)
    return missing

missing_train = check_image_paths(
    sampled_train,
    DATASET_DIR
)

missing_test = check_image_paths(
    sampled_test,
    DATASET_DIR
)

assert len(missing_train) == 0
assert len(missing_test) == 0

## Lưu metadata

Trước khi copy 20k ảnh mới, thì ta luuw metadat + list image_id trước


In [ ]:
TRAIN_OUTPUT = (
    SAMPLE_DIR /
    "Ebay_train_sample_10k.txt"
)

TEST_OUTPUT = (
    SAMPLE_DIR /
    "Ebay_test_sample_10k.txt"
)

def save_sop_metadata(df,output_path):
    df[
        [
            "image_id",
            "class_id",
            "super_class_id",
            "path"
        ]
    ].to_csv(
        output_path,
        sep=" ",
        header=False,
        index=False
    )

save_sop_metadata(
    sampled_train,
    TRAIN_OUTPUT
)
save_sop_metadata(
    sampled_test,
    TEST_OUTPUT
)

print(TRAIN_OUTPUT)
print(TEST_OUTPUT)

In [ ]:
# lưu thêm csv để sau này dễ thêm mấy cái trường dữ liệu khác
sampled_train.to_csv(
    SAMPLE_DIR / "train_10k.csv",
    index=False
)

sampled_test.to_csv(
    SAMPLE_DIR / "test_10k.csv",
    index=False
)

## Copy ảnh thành dataset 20k
Chỉ nên chạy khi cần tạo physical dataset.

In [ ]:
import shutil
from tqdm.auto import tqdm

def copy_sampled_images(df,dataset_root,output_root,split):
    split_root = output_root / split
    split_root.mkdir(
        parents=True,
        exist_ok=True
    )
    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Copying {split}"
    ):
        src = dataset_root / row["path"]
        # Preserve original directory structure
        dst = split_root / row["path"]
        dst.parent.mkdir(
            parents=True,
            exist_ok=True
        )
        shutil.copy2(src,dst)

COPY_IMAGES = True
if COPY_IMAGES:
    copy_sampled_images(
        sampled_train,
        DATASET_DIR,
        SAMPLE_DIR,
        "train"
    )
    copy_sampled_images(
        sampled_test,
        DATASET_DIR,
        SAMPLE_DIR,
        "test"
    )

## Final validation

In [ ]:
print("=" * 70)
print("FINAL DATASET VALIDATION")
print("=" * 70)

assert len(sampled_train) == TRAIN_SAMPLE_SIZE
assert len(sampled_test) == TEST_SAMPLE_SIZE
assert (
    len(sampled_train)
    +
    len(sampled_test)
    ==
    TOTAL_SAMPLE_SIZE
)
assert (
    len(
        set(sampled_train.class_id)
        &
        set(sampled_test.class_id)
    )
    == 0
)
assert (
    len(
        set(sampled_train.image_id)
        &
        set(sampled_test.image_id)
    )
    == 0
)

print("✓ Train size:", len(sampled_train))
print("✓ Test size :", len(sampled_test))
print("✓ Total     :", len(sampled_train) + len(sampled_test))
print("✓ Zero-shot class separation: PASS")
print("✓ Image separation: PASS")

## 4.Summary report

In [ ]:
summary = pd.DataFrame([
    {
        "split": "train",
        "original_images": len(train_df),
        "sampled_images": len(sampled_train),
        "original_classes": train_df.class_id.nunique(),
        "sampled_classes": sampled_train.class_id.nunique(),
        "class_coverage": train_coverage
    },
    {
        "split": "test",
        "original_images": len(test_df),
        "sampled_images": len(sampled_test),
        "original_classes": test_df.class_id.nunique(),
        "sampled_classes": test_df.class_id.nunique(),
        "class_coverage": test_coverage
    }
])

summary

## Conlcusion

The Stanford Online Products dataset is sampled independently within the original train and test splits.

Original:
- Train: 59,551 images / 11,318 classes
- Test: 60,502 images / 11,316 classes

Sampled:
- Train: 10,000 images
- Test: 10,000 images
- Total: 20,000 images

The sampling procedure:
1. Does not merge train and test.
2. Preserves the original train/test class separation.
3. Uses proportional stratified sampling according to the number of images available in each class.
4. Records class coverage after sampling.
5. Verifies that no class appears in both sampled train and test.
6. Saves the sampled metadata independently from the original dataset.

Important limitation:
Because the number of classes (22,634) is larger than the target number of images (20,000), it is mathematically impossible to retain at least one image from every original class.

Therefore, class coverage must be reported as an evaluation metric rather than assumed to be 100%.